# IC Pin Detection

In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s eta 0:00:00a 0:00:01


## Corrected the yaml configuration

In [5]:
import os
import yaml

# Define the root of all read-only inputs in Kaggle
search_path = '/kaggle/input'
val_images_path = None

print("Scanning filesystem for validation images...")

# Walk the directory tree to mathematically locate the 'val/images' folder
for root, dirs, files in os.walk(search_path):
    # Normalize path separators to ensure cross-platform string matching
    normalized_root = root.replace('\\', '/')
    if normalized_root.endswith('val/images'):
        val_images_path = normalized_root
        break

if val_images_path:
    # Calculate the root dataset directory (two hierarchical levels up from val/images)
    dataset_root = os.path.dirname(os.path.dirname(val_images_path))
    print(f"Success! Exact dataset root resolved to: {dataset_root}")
    
    # Construct the YOLO configuration dictionary
    config = {
        'path': dataset_root,
        'train': 'train/images',
        'val': 'val/images',
        'names': {0: 'pin'}
    }
    
    # Serialize the configuration to the writable working directory
    working_yaml_path = '/kaggle/working/data.yaml'
    with open(working_yaml_path, 'w') as file:
        yaml.dump(config, file, default_flow_style=False)
        
    print(f"Corrected data.yaml successfully written to: {working_yaml_path}")
    
else:
    print("CRITICAL ERROR: Could not locate 'val/images' anywhere in the /kaggle/input/ directory.")
    print("Please verify that your uploaded zip file contains the correct folder structure.")

Scanning filesystem for validation images...
Success! Exact dataset root resolved to: /kaggle/input/datasets/seikhmustakim/ic-pins/yolo_pins_dataset
Corrected data.yaml successfully written to: /kaggle/working/data.yaml


# Training code cells

In [ ]:
from ultralytics import YOLO

# Load the pretrained OBB nano model
model = YOLO('yolov8n-obb.pt')

# Train the model pointing to the root working directory configuration
results = model.train(
    data='/kaggle/working/data.yaml', # Corrected path
    epochs=100,
    imgsz=640,
    device=0  # This flag enforces GPU utilization
)

In [6]:
from ultralytics import YOLO

# Initialize the pretrained OBB nano model
model = YOLO('yolov8n-obb.pt')

# Execute the training loop optimized for extremely small datasets
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,               # High epoch count is safe due to early stopping
    imgsz=640,
    device=0,                 # Enforce GPU
    
    # --- Transfer Learning Constraints ---
    freeze=10,                # Freezes the first 10 layers (the backbone) to preserve generic feature extraction
    
    # --- Regularization & Early Stopping ---
    patience=25,              # Halts training if validation mAP plateaus
    weight_decay=0.001,       # Slightly increased L2 penalty for the un-frozen layers
    
    # --- Data Augmentation ---
    mosaic=1.0,               
    degrees=15.0,             
    fliplr=0.5,               
    flipud=0.5,               
    scale=0.5                 
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [13]:
import shutil

# Define the precise directory containing the weights and generated metrics
artifact_dir = '/kaggle/working/runs/obb/train-2'
output_zip = '/kaggle/working/ic_pins_deployment_assets'

# Compress the directory
print(f"Archiving execution assets from: {artifact_dir}")
shutil.make_archive(output_zip, 'zip', artifact_dir)

print(f"Success! Download {output_zip}.zip from the Kaggle Output panel.")
# The zip contains /weights/best.pt and metric graphs like F1_curve.png

Archiving execution assets from: /kaggle/working/runs/obb/train-2
Success! Download /kaggle/working/ic_pins_deployment_assets.zip from the Kaggle Output panel.
